# Experiment 19 — protected core with score and risk sizing

Based on [NB18](18-backtest-conditional-cash-allocation.ipynb) and the exact corrected-fee C-W benchmark from NB16/17.

**Goal:** test whether more than six positions increase net profit while controlling weaker additional holdings through position weights. Preserve C00's selection, entry6/exit8, regularised 90-day inverse-variance waterfall and source execution rules. C00 is a research benchmark, not deployed v11.

**Hypotheses:** extra breadth helps; performance score and risk jointly improve sizing; absolute tickets outperform normalised score × risk; holding units reduces turnover; eight/ten/twelve slots offer different economic trade-offs; findings survive smaller caps and a lower score floor.

**Method:** source-authoritative six-slot core; additional candidates retain source score/address order, without a rank cutoff. Compare flat, risk-only, score-only and combined tickets. Regularised daily variance uses 90 days, 30 actual-update shrinkage and a fixed eligible leave-one-out peer pool. Risk multiplier is `min(1, sqrt(peer prior / predicted variance))`; score multiplier is `0.5 + 0.5*clip(score,0,1)`, with missing scores using the 0.5 floor. Entry ticket is half the 5% single-position gross cap, multiplied before capacity clipping. Additional gross exposure is capped at 25%; unspent reductions remain cash. Hold means retaining shares, with source eligibility, hard caps and core-funding recalls. Resize and normalised controls use the same candidate order. Missing histories remain eligible when the causal peer risk prior is supported. Sparse NAV is carried forward after inception; actual updates and ages remain separate.

**Periods:** fresh \$150,000 accounts for **2026-01-01 → 2026-10-05 exclusive** and **2023-03-01 → 2026-10-08 exclusive**, with all earlier warmup including 2025. Two-day decisions and as-of recorded NAV. The last archive day is partial. 2026 is the default for tables/charts; full history appears separately. These searched, overlapping histories provide no untouched validation.

[Plan and formulas](19-protected-core-score-risk-allocation-plan.md) · [Claude review](19-protected-core-score-risk-allocation-review.md)

## Key new insights

Source-authoritative core parity passes. More positions are sized with absolute tickets rather than rewriting the core selector. Score/cap/trade-floor interactions determine whether score multipliers reach execution.

Resizing combined tickets RS01 gives a small profit/CAGR/Sharpe lead over C00 in both researched periods; holding those tickets loses in 2026. Combined score/risk holding also loses to risk-only holding in both periods, so this does not establish extra predictive value from the score multiplier. More slots and more deployment are not consistently more profitable.

## Summary of results

2026 C00: CAGR **58.59%**, Sharpe **2.47**, drawdown **-6.36%**. Combined held tickets RS00: **54.50%**, **2.32**, **-7.29%**, incremental net profit **\$-4,157**. Full-history results and every method appear below. [Detailed findings](19-protected-core-score-risk-allocation-findings.md).

RS01: **58.84% CAGR**, **2.48 Sharpe**, **-6.45% drawdown**, **\$+255** profit versus C00 in 2026. Full history: **33.66%** versus C00 **33.13%**, **\$+6,069** incremental profit. September cash remains **42.7%** in fresh 2026. The small 2026 gain comes with slightly worse drawdown.

## Robustness of results

Historical zero-extra complete equity/trade parity passes; common fresh-entry cached-tax reset applies to all economic arms. Recorded NAV, researched overlapping periods, retrospective guard/universe and inherited exit charges qualify conclusions. Contributor concentration, sparse observation support, costs, locked-slot outcomes and mechanical loss stress are reported separately. Independent C00 trajectories can diverge after sleeve PnL changes wealth.


In [1]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/'protected_core_score_allocation.py').exists():
    ROOT=ROOT/'scratchpad/hyperliquid-vov-2nd'
sys.path.insert(0,str(ROOT))
from protected_core_score_allocation import initialise,controls,run
from score_matrix_visualisation import compact_output


## Freeze repaired inputs

Verify inherited archive, strategy and framework identities before creating a separate NB19 artefact generation. All preceding indicator history remains available.

In [2]:
with compact_output('Pinned inputs; estimated 2–4 minutes'):
    data,evaluator=initialise()
from protected_core_score_checks import algebra
from IPython.display import Markdown,display
display(Markdown('### Formula and sparse-data checks — both registered periods\n\nIndependent fixtures verify tickets, clipped capacity, normalised budgets, missing risk and causal forward filling.'))
checks=algebra()
checks.to_csv(data['artifact']/'algebra-checks.csv',index=False)
display(checks.style.hide(axis='index'))


Pinned inputs; estimated 2–4 minutes — complete

### Formula and sparse-data checks — both registered periods

Independent fixtures verify tickets, clipped capacity, normalised budgets, missing risk and causal forward filling.

Check,Passed
"Worked absolute tickets: 3750, 1125, 1875",True
Weak-score reductions remain cash,True
Normalised worked allocation conserves cash,True
Normalised utility scale invariance,True
Capacity clipping after score/risk multipliers,True
Blocked incumbent can retain; no additional deposit,True
No pre-inception backfill,True
Sparse carried path produces variance,True
Prior-only risk multiplier is one,True
Unsupported risk stays missing,True


## Source authority and zero-extra parity

Replay historical C00 and the new adapter with no extra allocation for both periods. Require identical complete equity curves, trade times, quantities, prices and reserves before running treatments.

In [3]:
with compact_output('Historical controls and zero-slot replay; estimated 5–10 minutes'):
    control_runs=controls(data,evaluator)
from IPython.display import display,Markdown
display(Markdown('### Complete control parity — both registered periods\n\nExact equality checks for historical source C-W versus the zero-extra adapter. This establishes adapter parity before treatment trajectories diverge.'))
import pandas as pd
display(pd.read_csv(data['artifact']/'control-parity.csv').style.hide(axis='index'))


Historical controls and zero-slot replay; estimated 5–10 minutes — complete

### Complete control parity — both registered periods

Exact equality checks for historical source C-W versus the zero-extra adapter. This establishes adapter parity before treatment trajectories diverge.

Period,Equity_identical,Trades_identical,Trades
2026,True,True,473
Long,True,True,1044


## Execute registered independent accounts

Nine primary methods in each period, followed by the two registered sensitivities when combined fresh tickets are active. Every completed account and diagnostic ledger is saved. The source core targets are captured before extras and checked after merging; executable buys give the core priority.

In [4]:
with compact_output('Eighteen primary accounts and four sensitivities; estimated 15–35 minutes'):
    results,runs,attribution,absolute_weights=run(data,evaluator)


Eighteen primary accounts and four sensitivities; estimated 15–35 minutes — complete

## Economic outcomes and mechanism evidence

Read profit and risk alongside cash, actual funded breadth, ticket activation, turnover, costs and individual contributors. Every chart is displayed separately with its own heading and description.

In [5]:
from protected_core_score_report import report
with compact_output('Reporting and robustness; estimated 2–4 minutes'):
    findings=report(data,results,runs,attribution,absolute_weights,evaluator)


Reporting and robustness; estimated 2–4 minutes — complete

### Isolated source-state adapter cases — 2026-01-01 → 2026-10-05 exclusive (through 4 October, UTC)

The real additional-allocation merge is exercised with synthetic source states for unscored entry, blocked top-up, hard trim, minimum tickets, pinning, promotion and held-role isolation. These fixtures complement complete source replay parity; they do not validate asynchronous venue settlement.

Check,Passed
Unscored eligible entry receives positive floor ticket,True
Missing entry preserves source core target,True
Blocked incumbent retains exact share-value target without top-up,True
Hard trim below source band closes rather than breaches cap,True
Minimum-ticket failure does not create new position,True
Pinned source holding remains a real carry signal,True
Pinned value contributes to conservative equity bound,True
"Core promotion creates one core target, no extra target",True
Additional ownership at rank 6 cannot use core held-status bypass,True
Additional ownership at rank 7 cannot use core held-status bypass,True


<details><summary>Source, data and framework fingerprints</summary>

```json
{
  "identity": {
    "version": "nb19-v1",
    "inputs": {
      "version": "nb17-v1",
      "nb16_inputs": {
        "version": "nb15-v1",
        "nb14": {
          "version": "nb14-v1",
          "code": {
            "profitable_allocation.py": "a110b6debb07f9961f13e3ff56f9be634bacfd75ec9f98e3e7c8c3f7279d5a88",
            "profitable_allocation_report.py": "4b2587dfec7419fca290f598cca2cea1a64a85c4f2dcd20a5356d5d2cf9c255f",
            "create_profitable_allocation_notebook.py": "b41c7e4b2070eccf79e784f9df61b38d37f3b407b0969875563374a9dbac92ca"
          },
          "nb13_identity": {
            "source_fingerprint": "ea6a622a7706d49ddb818700e6b3d30ccdb412c41bfe0db2c9d1187515c44cdf",
            "extended_universe_fingerprint": "67bffc9ed7d5997250513dd6fba3e00cbc275f91ed65300664a6cc31c49fad1e",
            "feature_panel": "f7e29b5e36a30531933bc2e188dfa30c99e29253b3ccd7071e84f1b0dee35b73",
            "code": {
              "long_candidate_backtest.py": "1c60331cab792097a436f2e2c0f1d8c6e87f98aabc51a1a633e3c3d98347b1ff",
              "score_matrix.py": "e3ca6d6bf07f418121dabf91aa6e21d5a9eeaf5cc4bb86120c233e4db089a503",
              "allocation_followup.py": "eb2513a3d8d0e2e7d6d0cc447c700705d3ec1f01a954d462c898288898d07b87",
              "allocation_followup_engine.py": "0951f17d4c7ce3af6f3bca2b361d439057535ae12c0433ae006c772aae425048",
              "portfolio_snapshot_risk.py": "d6a10069cf851d9504eb50fd6698cd05fd3de109d8a91ad76d6ffada4c73b854",
              "sizing_study_data.py": "fa07af5ed0b936be2fa43ebe410858d28295673554cff0fdda7cedcbded33b1f"
            },
            "shortlist": [
              {
                "Candidate": "C01",
                "Name": "Legacy 45d / 4 inputs / 6m Sortino + 2m CAGR / 6 positions",
                "Positions": 6.0,
                "Kind": "score",
                "Score name": "Legacy 45d / 4 inputs / 6m Sortino + 2m CAGR",
                "Policy": ""
              },
              {
                "Candidate": "C02",
                "Name": "Blend 46d / 3 inputs / 6m CAGR / 6 positions",
                "Positions": 6.0,
                "Kind": "score",
                "Score name": "Blend 46d / 3 inputs / 6m CAGR",
                "Policy": ""
              },
              {
                "Candidate": "C03",
                "Name": "Legacy 45d / 4 inputs / 6m CAGR + 1m Sortino / 6 positions",
                "Positions": 6.0,
                "Kind": "score",
                "Score name": "Legacy 45d / 4 inputs / 6m CAGR + 1m Sortino",
                "Policy": ""
              },
              {
                "Candidate": "C04",
                "Name": "Blend 46d / 7 positions",
                "Positions": 7.0,
                "Kind": "score",
                "Score name": "Blend 46d",
                "Policy": ""
              },
              {
                "Candidate": "C05",
                "Name": "Legacy 45d / 4 inputs / 2m CAGR + 6m Sharpe / 6 positions",
                "Positions": 6.0,
                "Kind": "score",
                "Score name": "Legacy 45d / 4 inputs / 2m CAGR + 6m Sharpe",
                "Policy": ""
              },
              {
                "Candidate": "C06",
                "Name": "Blend 46d / 4 inputs / 6m Sortino + 2m CAGR / 7 positions",
                "Positions": 7.0,
                "Kind": "score",
                "Score name": "Blend 46d / 4 inputs / 6m Sortino + 2m CAGR",
                "Policy": ""
              },
              {
                "Candidate": "C07",
                "Name": "NB12 S18 F:gap_log L90/p2/\u03ba30",
                "Positions": 6.0,
                "Kind": "allocation",
                "Score name": "Legacy 45d / 4 inputs / 6m CAGR + 1m Sortino",
                "Policy": "S18"
              },
              {
                "Candidate": "C08",
                "Name": "NB12 X01 F:regular L75/p2/\u03ba30",
                "Positions": 6.0,
                "Kind": "allocation",
                "Score name": "Legacy 45d / 4 inputs / 6m CAGR + 1m Sortino",
                "Policy": "X01"
              },
              {
                "Candidate": "C09",
                "Name": "Legacy 45d / 4 inputs / 2m CAGR + 6m Sharpe / 7 positions",
                "Positions": 7.0,
                "Kind": "score",
                "Score name": "Legacy 45d / 4 inputs / 2m CAGR + 6m Sharpe",
                "Policy": ""
              },
              {
                "Candidate": "C10",
                "Name": "NB12 S17 F:calendar_log L90/p2/\u03ba30",
                "Positions": 6.0,
                "Kind": "allocation",
                "Score name": "Legacy 45d / 4 inputs / 6m CAGR + 1m Sortino",
                "Policy": "S17"
              }
            ],
            "shortlist_data": {
              "07/2026-results.csv": "860cbacb9fa78a6281276d72a28f9ab08c18610d12368efb8c422d59e6f8ff1f",
              "12/465e3b8cf13b49e11f635d12c951268318e3b2c39e120f25b89190dd66606be5/engine_summary.parquet": "bb8fabffd0d61bfe4eab10a392873e49cf3fd4a1661439f41082247371909805"
            },
            "downloads": {
              "vault_prices": {
                "path": "/Users/moo/code/getting-started/scratchpad/hyperliquid-vov-2nd/_artifacts-restored-2026-10-07/inputs/vault-prices.parquet",
                "sha256": "30ec7e9dfa7eb236518cf1922a9ba82f4f42cfbfb041c06f5db3273d114aa8f2",
                "bytes": 289315157
              },
              "vault_metadata": {
                "path": "/Users/moo/code/getting-started/scratchpad/hyperliquid-vov-2nd/_artifacts-restored-2026-10-07/inputs/vault-metadata.json",
                "sha256": "6c5ec12de41f1c10fc48e54a832eff065d14c77ea19c85fd482c46ea33a5382c",
                "bytes": 83587673
              },
              "permission_sidecar_status": {
                "http_status": 404,
                "used": false,
                "note": "Source strategy uses permission projection; independent endpoint returned 404"
              }
            },
            "start": "2023-03-01 00:00:00",
            "end": "2026-10-08 00:00:00",
            "control_arms": [
              "v11",
              "regular90"
            ]
          },
          "arms": {
            "A": [
              "Source variance / original allocation",
              "source",
              "sequential"
            ],
            "B": [
              "Source variance / simultaneous allocation",
              "source",
              "simultaneous"
            ],
            "C": [
              "Regularised variance / original allocation",
              "regular",
              "sequential"
            ],
            "D": [
              "Regularised variance / simultaneous allocation",
              "regular",
              "simultaneous"
            ],
            "G": [
              "Regularised downside / original allocation",
              "downside",
              "sequential"
            ]
          },
          "periods": {
            "2026": [
              "2026-01-01 00:00:00",
              "2026-10-05 00:00:00"
            ],
            "Long": [
              "2023-03-01 00:00:00",
              "2026-10-08 00:00:00"
            ]
          },
          "pricing": "as_of_observation",
          "sensitivity": "fresh_quote_within_source_two_day_cycle",
          "initial_cash": 150000
        },
        "source": "/Users/moo/code/strategies/strategy/hyper-ai-v11.py",
        "source_sha256": "5ed8a4c00f1edb14ae9af40ac3d94d3622587445921d1e92ef33f9676761ffbe",
        "code": {
          "allocation_comparison.py": "a7b804e4e4d214d91eee77ee764f914fee998143777065c25809bddaa6db4c76",
          "create_allocation_comparison_notebook.py": "e603ce4edfdf364699177337507baafacef319e4a3777f923a2eba0873430081"
        },
        "arms": [
          "V11",
          "A",
          "C",
          "G"
        ],
        "initial_cash": 150000,
        "periods": {
          "2026": [
            "2026-01-01 00:00:00",
            "2026-10-05 00:00:00"
          ],
          "Long": [
            "2023-03-01 00:00:00",
            "2026-10-08 00:00:00"
          ]
        },
        "pricing": "as_of_observation"
      },
      "arms": {
        "M00": "Exact C-W benchmark",
        "M01": "Admission-aware risk waterfall",
        "M02": "Proportional capped risk",
        "M03": "Score-priority waterfall",
        "M04": "Core + two score-first overflow",
        "M05": "Core + four score-first overflow",
        "M06": "Core + four scored-only overflow",
        "M07": "Capacity-aware incumbent retention",
        "M08": "Greedy score-dollar basket",
        "M09": "Maintained / fee-aware overflow",
        "M10": "One-decision rank-exit confirmation",
        "M11": "Joint six-slot score optimiser",
        "M12": "Legacy capacity extension",
        "M13": "Fixed ten-name core"
      },
      "numerical": {
        "dollar_tolerance": 1e-06,
        "objective_tolerance": 1e-10,
        "mip_rel_gap": 1e-08,
        "time_limit": 30.0
      },
      "code": {
        "capital_allocation.py": "7e4b90c272d628eb3a114aaa0efeff93e18972df0914a9fa818dcc77069a27ab",
        "capital_allocation_methods.py": "9c54723c01be9a0ada347842ddb254717b032703afc02f3ba2c6bcc5aad234b3"
      },
      "source_sha256": "5ed8a4c00f1edb14ae9af40ac3d94d3622587445921d1e92ef33f9676761ffbe"
    },
    "methods": {
      "C00": "Exact corrected C-W",
      "E00": "Flat tickets / hold units",
      "R00": "Risk tickets / hold units",
      "S00": "Score tickets / hold units",
      "RS00": "Score \u00d7 risk / hold units",
      "RS01": "Score \u00d7 risk / resize",
      "NRS00": "Normalised score \u00d7 risk / resize",
      "RS08": "Score \u00d7 risk / eight slots",
      "RS12": "Score \u00d7 risk / twelve slots",
      "GF25": "Score floor 0.25 / hold",
      "CAP10": "Smaller exposure caps / hold"
    },
    "registry": {
      "risk_days": 90,
      "actual_update_shrinkage": 30,
      "score_floor": 0.5,
      "single_gross_cap": 0.05,
      "total_gross_cap": 0.25,
      "ticket_fraction": 0.5,
      "core_slots": 6,
      "entry_rank": 6,
      "exit_rank": 8,
      "missing": true,
      "initial_cash": 150000
    },
    "code": "ad815fcb39ab3bac775434f02bf6fff5429dce2efabd1b08ef804c5a926f76cc"
  },
  "framework": {
    "getting-started": "0be3392775ffd2567a617153a3c8d42fb0e57476",
    "trade-executor": "2db909b32d854d342f159074b2b92948f32b69d6",
    "trading-strategy": "unavailable",
    "web3-ethereum-defi": "unavailable"
  },
  "periods": {
    "2026": [
      "2026-01-01 00:00:00",
      "2026-10-05 00:00:00"
    ],
    "Long": [
      "2023-03-01 00:00:00",
      "2026-10-08 00:00:00"
    ]
  },
  "limitations": [
    "Researched overlapping histories, retrospective universe/guard",
    "NAV observation timestamp assumed available; executable NAV unverified",
    "Inherited 10% profit and 0.1% capital fee allowances",
    "Same-state core intentions are protected; future wealth and core membership can change"
  ]
}
```

</details>

### Strategy logic comparison — 2026-01-01 → 2026-10-05 exclusive (through 4 October, UTC)

Frozen core score/selector/waterfall in all methods. Only additional ticket utility, management and slot/exposure limits vary. Missing inputs retain positive score floors and supported peer risk.

...

### Leading contributors and fixed-ledger removal — 2023-03-01 → 2026-10-08 exclusive (through 7 October, UTC; final archive day partial)

Removing the largest positive contributor is a descriptive robustness check without rebalancing or changing later membership. It is not a replay of a tradable replacement strategy.

Arm,Top_vault,Largest contribution (USD),Profit after deleting largest (USD)
C00,Delta_01,"$54,237","$216,645"
E00,Delta_01,"$54,223","$221,183"
R00,Delta_01,"$54,223","$219,904"
S00,Delta_01,"$54,224","$220,772"
RS00,Delta_01,"$54,224","$218,755"
RS01,Delta_01,"$54,234","$222,718"
NRS00,Delta_01,"$54,214","$215,814"
RS08,Delta_01,"$54,229","$219,892"
RS12,Delta_01,"$54,211","$217,976"
GF25,Delta_01,"$54,225","$218,199"


### Best-update and quarter deletion sensitivity — 2023-03-01 → 2026-10-08 exclusive (through 7 October, UTC; final archive day partial)

Remove one booked daily update or one calendar quarter from each fixed account ledger. No reinvestment or replacement is simulated; these dollar sensitivities are not recalculated CAGR or independent validation. Partial quarters remain included.

Arm,Profit excluding best daily update (USD),Lowest profit after excluding one quarter (USD),Highest profit after excluding one quarter (USD)
C00,"$243,250","$192,124","$283,404"
E00,"$247,774","$196,648","$287,928"
R00,"$246,495","$195,369","$286,649"
S00,"$247,363","$196,237","$287,517"
RS00,"$245,348","$194,221","$285,502"
RS01,"$249,319","$198,193","$289,474"
NRS00,"$242,395","$191,269","$282,550"
RS08,"$246,489","$195,363","$286,643"
RS12,"$244,555","$193,428","$284,709"
GF25,"$244,792","$193,666","$284,947"


## Verdict and next step



- **H1 — Extra breadth:** E00 − C00 \$-4,125 in 2026 / \$+4,524 full history; mixed across histories. These are package comparisons, not independent statistical confirmations.
- **H2 — Combined score/risk versus risk alone:** RS00 − R00 \$-136 in 2026 / \$-1,147 full history; rejected on both inspected histories. These are package comparisons, not independent statistical confirmations.
- **H3 — Absolute versus normalised resizing:** RS01 − NRS00 \$+2,381 in 2026 / \$+6,924 full history; supported on both inspected histories. These are package comparisons, not independent statistical confirmations.
- **H4 — Hold versus resize:** RS00 − RS01 \$-4,411 in 2026 / \$-3,972 full history; rejected on both inspected histories. These are package comparisons, not independent statistical confirmations.
- **H5 — Twelve versus eight slots:** RS12 − RS08 \$-732 in 2026 / \$-1,935 full history; rejected on both inspected histories. These are package comparisons, not independent statistical confirmations.

**H6 — local assumptions:** lower floor GF25 − RS00 profit \$-78 / \$-555; smaller-cap CAP10 − RS00 \$+2,722 / \$-3,627. Neither floor nor caps are established as optimal.

**Practical shortlist:** RS01 leads inspected 2026 CAGR. RS01 is the direct management contrast to the primary held-ticket method. Compare its full-history profit, drawdown, contributors and costs before considering further observations. More cash use alone is not a reason to promote a strategy.

**Economic attribution:** RS01 gains only \$+255 in fresh 2026. Interval-start role bookkeeping assigns \$-189 to additional holdings and \$+444 to core/other feedback. This does not show profitable extras in isolation. Its September cash is 42.7%, versus 49.8%; substantial cash remains.

**Timing and concentration:** economic accounts are identical before the first additional opportunities in late 2025 in the long account. The fresh-2026 common strongest update is 26 June: Realist Capital and Goon Edging each gain about 14% over roughly two observed days while BTC/ETH fall about 4.7%/6.0%. Realist contributes about 35% of positive 2026 vault profit. The long strongest update, 13 November 2024, precedes any additional allocation and therefore cannot explain a treatment advantage. These are sparse recorded-NAV updates, not demonstrated executable fills at every intervening timestamp.

## Robustness of results

Complete historical versus zero-extra equity and trade parity is required. The source core targets are unchanged by the merge and executed gross exposure/slot checks pass. A common fresh-entry cached-tax reset isolates the previously identified close/reopen marking inconsistency; historical control metrics are shown separately. Daily-resampled Sharpe includes off-cycle zero bins.

Contributor deletion, strongest-update curator/BTC/ETH context, overlapping quarter differences, sparse actual-update support, locked-slot counterfactuals and mechanical sleeve loss stress appear in the notebook. The hard-cap audit caught and repaired a suppressed trim: marked-to-gross conversion now uses each holding’s actual ratio, with a full close when the required reduction is below the source band. All accounts were rerun under the new identity. The conservative denominator includes both existing-position liquidation allowances and a possible new-entry capital allowance; initial all-cash snapshots are checked as well. No untouched holdout exists. Recorded NAV is not proof of an executable quote, and the inherited fee assumptions remain economically unverified.

## Next action

Treat improved historical profit and risk as a shortlist criterion. Require a later observation period and verified fee/permission conventions before production changes. Do not optimise floor, slot count or exposure limits using this same result table.
